# `ask_anything()` — the capability typology as one executable function

One field, any question. The system decides the bucket and answers **only in the way that
bucket can be answered faithfully**:

```
ask_anything(question)
 |- router: aggregate form?  no  -> BUCKET 1  retrieval -> grounded answer + citations
 `- yes -> which column does the answer need?
     |- validated CONTENT column (alienation_alleged)  -> BUCKET 3 (extracted):
     |       threshold-aware count + abstention audit
     |- METADATA columns                                -> BUCKET 2: NL->SQL with guardrails
     |       (coder model, few-shot, temp 0, EXPLAIN-validated, SQL printed for review)
     `- no column carries the answer                    -> BUCKET 3 (unextracted): refusal
```

Nothing here is new machinery: the router, retrieval, DuckDB layer, threshold helper **and
the NL->SQL translator** are exec'd from `rag_echr_ris.ipynb` and `echr_query.ipynb`.
The metadata branch is genuinely semantic — the local coder model maps "against Poland" to
`respondent_state = 'POL'` and "Kanton Bern" to the Swiss table itself — behind **two
deterministic refusal nets**: a deny-list of known-unextracted content concepts (semantic
knowledge no validator can derive) and `EXPLAIN` schema validation of every generated
statement. Refusal is never delegated to the model — measured: a 3B model offered a
CANNOT_ANSWER escape takes it whenever the question needs composition. The generated SQL
always prints **above** its result: for aggregate answers the trust boundary is the query
text, not the number. The citable quantitative path remains the reviewed canned queries.

## 1. Load the deployed pipeline (RAG cells + query-layer cells, cache-hot)

In [6]:
import json
from pathlib import Path

RAG_NB   = Path("rag_echr_ris.ipynb")
QUERY_NB = Path("echr_query.ipynb")

def exec_cells(nb_path, markers):
    nb = json.loads(nb_path.read_text())
    n = 0
    for c in nb["cells"]:
        if c["cell_type"] != "code":
            continue
        src = "".join(c["source"])
        if any(m in src for m in markers):
            exec(src, globals())
            n += 1
    print(f"exec'd {n} cells from {nb_path.name}")

# RAG pipeline: config, loaders, chunker, genre, corpus, embedder/index, retrieve,
# router (Layer 1), generation + answer()
exec_cells(RAG_NB, ["DATA_DIR  = Path", "def load_json_records", "ECHR_ANCHORS = [",
                    "reusable ECHR genre helper", "records, chunks = [], []",
                    "_embedder = None", "def _to_hit", "AGGREGATE_PATTERNS = [",
                    "SYSTEM_PROMPT = ("])
assert index is not None, "index missing -- run rag_echr_ris.ipynb once first"

# Query layer: config, table -> DuckDB, threshold-aware helper, NL->SQL translator
exec_cells(QUERY_NB, ["MIN_CONFIDENCE = 0.70", "def _load_table", "def alienation_at(",
                      "def nl2sql"])
assert con is not None, "DuckDB table missing -- run echr_extraction + theme classify first"

print(f"ready: {index.ntotal} chunks | {len(df)} table rows | router: {len(AGGREGATE_PATTERNS)} patterns")

inputs: ['echr_parental_alienation.json', 'ris_parental_alienation.json', 'swiss_parental_alienation.json']
chunk=600w/80o | ECHR skip={'OPINION', 'RELEVANT_LAW'} law_only=False | dev_cap=None
RIS decisions=True (civil only) | Swiss=True
genre routing: exclude={'communicated'} | MMR fetch_k=30 lambda=0.7
normalisers ready: ['echr', 'ris', 'swiss']
section splitter + chunker ready
genre helper ready: ('merits', 'admissibility', 'communicated', 'other') | low-info: {'communicated'}
  echr : 1116 records from echr_parental_alienation.json
  ris : 38 principles + 479 civil decisions (dropped 31 criminal-senate/AUSL 'Text' records — Entfremdung homonym / ECtHR summaries)
  ris  : 517 records from ris_parental_alienation.json
  swiss: dropped 24 Rechenschaftsbericht records (court annual reports, not case law — multi-case digests that flood the top-k)
  swiss: 2007 records from swiss_parental_alienation.json

ECHR dates: 1114/1116 have YYYY-MM-DD, 2 n.d.
ECHR sections: 873/1116 parsed; 243 f

## 2. Metadata relations for the Bucket-2 handlers
Quiet re-registration of the cross-jurisdiction metadata (same fields as `echr_query.ipynb`
section 6b; the coverage audit lives there).

In [ ]:
import json as _json
import re as _re
import pandas as pd

_swiss = _json.loads((DATA_DIR / "swiss_parental_alienation.json").read_text())
_ris   = _json.loads((DATA_DIR / "ris_parental_alienation.json").read_text())

swiss_meta = pd.DataFrame([{"id": r.get("stable_id"), "canton": r.get("canton"),
                            "court_type": r.get("court_type"), "year": r.get("year")}
                           for r in _swiss])

_SEN = _re.compile(r"\d{1,3}\s*([A-Z][a-z]{1,2})")

def _senate(gz):
    m = _SEN.match((gz or "").split(";")[0].strip())
    return m.group(1) if m else None

ris_meta = pd.DataFrame([{
    "id": r.get("id"), "dokumenttyp": r.get("dokumenttyp"),
    "year": int(r["entscheidungsdatum"][:4]) if (r.get("entscheidungsdatum") or "")[:4].isdigit() else None,
    "senate": _senate(r.get("geschaeftszahl")),
} for r in _ris])

# ECHR metadata incl. the 2026-07-07 fields (formation, introduction->judgment duration)
from datetime import datetime as _dt
_echr_raw = _json.loads((DATA_DIR / "echr_parental_alienation.json").read_text())

def _pdate(s):
    s = (s or "").split()[0] if s else ""
    try:
        return _dt.strptime(s, "%d/%m/%Y").date()
    except (ValueError, IndexError):
        return None

def _formation(dc):
    for f in ("GRANDCHAMBER", "CHAMBER", "COMMITTEE"):
        if f in (dc or ""):
            return f
    return None

def _dur(r):
    if (r.get("doctype") or "").upper() == "HECOM":
        return None                      # communicated = still pending, no duration
    a = _pdate(r.get("introductiondate"))
    b = _pdate(r.get("judgementdate")) or _pdate(r.get("decisiondate"))
    return (b - a).days if a and b and b >= a else None

# extraction-layer dates (echr table): HUDOC metadata leaves most HECOMs undated
# (36/238 have judgementdate); the extraction layer parsed the communication date
# from the document text (236/238). Reuse it as the last fallback, same convention.
_extr_year = {r.id: int(str(r.judgment_date)[:4]) for r in df.itertuples()
              if str(r.judgment_date)[:4].isdigit()}

def _year(r):
    """ECLI year when judged; else HUDOC date fields; else extraction-layer date.
    For HECOM (no ECLI until judged) the year is the COMMUNICATION year."""
    e = r.get("ecli") or ""
    if e.startswith("ECLI:CE:ECHR:") and e.split(":")[3][:4].isdigit():
        return int(e.split(":")[3][:4])
    d = _pdate(r.get("judgementdate")) or _pdate(r.get("decisiondate"))
    return d.year if d else _extr_year.get(r.get("itemid"))

echr_meta = pd.DataFrame([{
    "id": r.get("itemid"), "respondent": r.get("respondent"),
    "importance": int(r["importance"]) if str(r.get("importance", "")).isdigit() else None,
    "year": _year(r),
    "formation": _formation(r.get("documentcollectionid")),
    "duration_days": _dur(r),
    "separate_opinion": str(r.get("separateopinion", "")).upper() == "TRUE",
} for r in _echr_raw])
echr_kp = pd.DataFrame([
    {"id": r.get("itemid"), "kp_code": code.strip()}
    for r in _echr_raw for code in (r.get("kpthesaurus") or "").split(";") if code.strip()
])

con.register("swiss_meta", swiss_meta)
con.register("ris_meta", ris_meta)
con.register("echr_meta", echr_meta)
con.register("echr_kp", echr_kp)

# make the German corpora visible to the NL->SQL translator
SCHEMA = SCHEMA + (
    " Additional tables: swiss_meta(id TEXT, canton TEXT two-letter Swiss canton e.g. "
    "ZH=Zuerich, BE=Bern, AG=Aargau, BL=Basel-Land, BS=Basel-Stadt, GR=Graubuenden, "
    "SG=St.Gallen, CH=federal, court_type TEXT, year INT) = Swiss decisions; "
    "ris_meta(id TEXT, dokumenttyp TEXT[Rechtssatz|Text], year INT, senate TEXT) "
    "= Austrian OGH records.")
FEW_SHOT = FEW_SHOT + [
    ("Aus welchem Kanton stammen die meisten Schweizer Entscheidungen?",
     "SELECT canton, COUNT(*) AS n FROM swiss_meta GROUP BY canton ORDER BY n DESC LIMIT 5;"),
]
# ---- dynamic content fields: discover anything field_deploy.ipynb has shipped ----
# each deployed field = data/field_<name>_deployed.parquet + data/field_<name>_meta.json
# (definition, lexicon, threshold, validation). Registration + routing need NO code changes:
# define -> review -> deploy -> queryable here.
import glob as _glob
DEPLOYED_FIELDS = {}
for _mp in sorted(_glob.glob(str(DATA_DIR / "field_*_meta.json"))):
    _meta = _json.loads(open(_mp).read())
    _fname = _meta["field"]
    _pq = DATA_DIR / f"field_{_fname}_deployed.parquet"
    if not _pq.exists():
        continue
    _df = pd.read_parquet(_pq)
    con.register(f"field_{_fname}", _df)
    DEPLOYED_FIELDS[_fname] = _meta
    SCHEMA = SCHEMA + (f" Table field_{_fname}(id TEXT, {_fname} BOOLEAN, conf_cal DOUBLE): "
                       f"calibrated content field, genuine = {_fname} AND conf_cal >= "
                       f"{_meta['threshold']}.")
if DEPLOYED_FIELDS:
    print("deployed content fields discovered:",
          {k: v["validation"] for k, v in DEPLOYED_FIELDS.items()})

print(f"registered swiss_meta ({len(swiss_meta)}) + ris_meta ({len(ris_meta)}) + "
      f"echr_meta ({len(echr_meta)}, year coverage "
      f"{echr_meta.year.notna().mean()*100:.0f}%, duration coverage "
      f"{echr_meta.duration_days.notna().mean()*100:.0f}%) + echr_kp ({len(echr_kp)}) | "
      f"schema + few-shot extended")

## 3. The dispatcher — content column, deny-list, then guarded NL→SQL
Order matters and each step is the *cheapest sufficient* mechanism:

1. **Validated content column** (`alienation_alleged`) — keyword-routed, because it must reach
   the calibrated-confidence machinery, and there is exactly one such column.
2. **Deny-list of known-unextracted concepts** (custody outcome, marital status, duration) —
   *semantic* knowledge about what is NOT in the data; no SQL validator can know it, and
   (measured) a 3B model given a refusal escape over-uses it on composable questions, so the
   model is never asked to refuse.
3. **Guarded NL→SQL** for everything else — the local coder model maps entities itself
   ("against Poland" → `respondent_state='POL'`, "Kanton Bern" → `swiss_meta`, canton `BE`),
   behind `EXPLAIN` validation with one retry. The generated SQL prints above its result —
   review the query, not the number.

Failure direction of every net is **refusal, never fabrication**.

In [ ]:
def _h_alienation(q):
    """BUCKET 3 (extracted): the one validated content column, threshold-aware."""
    res, audit = alienation_at(min_confidence=MIN_CONFIDENCE, mode="filter")
    print("  BUCKET 3 (content aggregate -- extracted + calibrated)")
    print(f"  ANSWER: {audit['passed']} cases with a confident alienation allegation "
          f"(calibrated conf >= {audit['min_confidence']}); "
          f"{audit['abstained_low_conf']} further predicted-positive cells ABSTAINED "
          f"(low confidence, surfaced not dropped); basis={audit['confidence_basis']}")
    print(res[["id", "title", "respondent_state", "conf_eff"]].head(5).to_string(index=False))
    print("  CAVEAT: extractor validated at F1 0.51-0.59 on 120 gold labels; ECHR Article-8 "
          "English cases only -- a calibrated estimate, not a hard fact.")


REFUSAL = ("BUCKET 3 (content aggregate -- NOT extracted): no validated per-case column "
           "carries this answer (e.g. custody outcome, marital status). "
           "Answering from retrieval would fabricate a statistic; the faithful options are "
           "building + validating an extractor for that field (see echr_extraction.ipynb) "
           "or this refusal.")

# net 1 -- known-unextracted content concepts (semantic knowledge SQL validation cannot have):
_UNEXTRACTED = _re.compile(
    r"(receiv\w*|erhielt|erhält|awarded|granted|zugesprochen)\s+(the\s+)?(sole\s+)?"
    r"(custody|sorgerecht|obhut)"
    r"|(custody|sorgerecht|obhut)\s+(was\s+)?(receiv|award|grant|zugesprochen)"
    r"|\bmarried\b|\bverheiratet\b",
    _re.IGNORECASE)   # duration removed 2026-07-07: introductiondate arrived,
                      # proceedings duration moved from refusal to Bucket 2

MAX_RESULT_ROWS = 40  # display cap for Bucket-2 results; truncation is announced, never silent


def _coverage_note(sql):
    """Transparency net: for every table.column the generated SQL touches, report
    rows with no value there. A question phrased against an incomplete column
    silently drops those rows from filters and groupings -- the user must see that."""
    tables = set(_re.findall(r"\b(?:FROM|JOIN)\s+([A-Za-z_]\w*)", sql, _re.IGNORECASE))
    sql_words = {w.lower() for w in _re.findall(r"[A-Za-z_]\w*", sql)}
    for t in sorted(tables):
        try:
            cols = [r[0] for r in con.execute(f"DESCRIBE {t}").fetchall()]
        except Exception:
            continue
        for c in cols:
            if c.lower() not in sql_words or c.lower() == "id":
                continue
            n, filled = con.execute(f'SELECT COUNT(*), COUNT("{c}") FROM {t}').fetchone()
            if filled < n:
                print(f"  COVERAGE: {t}.{c} has a value in {filled}/{n} rows -- the other "
                      f"{n - filled} rows are invisible to any filter or grouping on it.")


def _h_nl2sql(q):
    """BUCKET 2 (metadata aggregate) via guarded NL->SQL. Two deterministic nets:
    deny-list (above, checked inside nl2sql too) and EXPLAIN validation + retry.
    Refusal is never delegated to the model (measured lazy-escape effect)."""
    sql = nl2sql(q)
    if sql is None:
        print("  BUCKET 2 (metadata aggregate) -- UNAVAILABLE, not refused:")
        print("  the local NL->SQL translator is unreachable (start `ollama serve` and re-run).")
        print("  The question IS answerable from metadata; see the canned queries in echr_query.ipynb.")
        return
    if sql.startswith("--"):
        print(f"  NL->SQL declined/invalid: {sql}")
        print(" ", REFUSAL); return
    print("  BUCKET 2 (metadata aggregate) -- generated SQL (REVIEW THIS, it is the trust boundary):")
    print("   ", sql)
    try:
        out = run_sql(sql)
        print(out.head(MAX_RESULT_ROWS).to_string(index=False))
        if len(out) > MAX_RESULT_ROWS:
            print(f"  ... {len(out) - MAX_RESULT_ROWS} more rows not shown "
                  f"(display cap {MAX_RESULT_ROWS}; add LIMIT/ORDER BY to the question to narrow)")
    except Exception as e:
        print("  execution failed:", e); print(" ", REFUSAL); return
    _coverage_note(sql)
    print("  CAVEAT: counts describe the keyword-matched corpora, never litigation rates; "
          "citable numbers come from the reviewed canned queries in echr_query.ipynb.")


def _match_deployed_field(question):
    """Route a question to a deployed content field when its lexicon (or name) appears."""
    q = question.lower()
    for fname, meta in DEPLOYED_FIELDS.items():
        terms = [fname.replace("_", " ")] + [t for t in meta.get("lexicon", []) if len(t) > 5]
        if any(t.lower() in q for t in terms):
            return fname
    return None


def _h_deployed_field(fname):
    """BUCKET 3 (extracted + calibrated) for a factory-deployed field — generic handler."""
    meta = DEPLOYED_FIELDS[fname]
    thr = meta["threshold"]
    res = run_sql(f"""SELECT COUNT(*) FILTER ({fname} AND conf_cal >= {thr}) AS confident,
                             COUNT(*) FILTER ({fname} AND conf_cal <  {thr}) AS abstained,
                             COUNT(*) AS corpus
                      FROM field_{fname}""")
    conf_n, abst_n = int(res.confident[0]), int(res.abstained[0])
    v = meta["validation"]
    print(f"  BUCKET 3 (content aggregate -- extracted + calibrated, field `{fname}`)")
    print(f"  ANSWER: {conf_n} cases with a confident {fname.replace('_', ' ')} "
          f"(calibrated conf >= {thr}); {abst_n} further predicted-positive cells ABSTAINED "
          f"(low confidence, surfaced not dropped)")
    top = run_sql(f"""SELECT f.id, e.title, f.conf_cal FROM field_{fname} f
                      LEFT JOIN echr e ON f.id = e.id
                      WHERE f.{fname} AND f.conf_cal >= {thr}
                      ORDER BY f.conf_cal DESC LIMIT 5""")
    print(top.to_string(index=False))
    print(f"  CAVEAT: LLM extractor validated on {v['n_labels']} reviewed labels "
          f"(F1 {v['llm_f1']}, flip rate {v['flip_rate']}); "
          f"domain of validity: {meta['corpus'].split(' ')[0]}.")


def ask_anything(question, k=4):
    print("=" * 88)
    print("Q:", question)
    trig = aggregate_trigger(question)
    if not trig:
        print("ROUTE: Bucket 1 (no aggregate trigger) -> retrieval + grounded generation")
        r = answer(question, k=k)
        if r["abstained"]:
            print(f"  abstained: {r['abstain_type']} -- {r['answer'][:200]}")
        elif r["answer"]:
            print("  ANSWER:", r["answer"][:600].replace("\n", " "))
        else:
            print("  (generation OFF -- LLM unreachable; retrieval-only mode, hits below)")
        for n, h in enumerate(r["hits"][:3], 1):
            print(f"  [{n}] cos={h['score']:.3f} | {h['jurisdiction']:9s} | {h['title'][:55]}")
        return
    print(f"ROUTE: aggregate trigger '{trig}' -> query layer (never generation)")
    if _re.search(r"alienat|entfremd", question, _re.IGNORECASE):
        _h_alienation(question)                      # validated content column first
        return
    fld = _match_deployed_field(question)            # factory-deployed fields, auto-routed
    if fld:
        _h_deployed_field(fld)
    elif _UNEXTRACTED.search(question):
        print(" ", REFUSAL)                          # known-unextracted concept
    else:
        _h_nl2sql(question)                          # semantic metadata path


print("ask_anything() ready -- dispatch: content column -> deny-list -> guarded NL->SQL")

## 4. One entry point, seven questions, four distinct behaviours
Two Bucket-1 (answered with citations, one DE one EN), one aggregate in each metadata
dimension, the extracted content aggregate with its abstention audit, and one question the
system **correctly refuses**.

In [9]:
DEMO = [
    "When can custody be transferred to the other parent because of alienating behaviour?",
    "Unter welchen Voraussetzungen kann einem Elternteil die Obhut entzogen werden?",
    "How many cases against Poland are in the corpus?",              # NOT in few-shot
    "Wie viele Schweizer Entscheidungen stammen aus dem Kanton Bern?",  # NOT in few-shot
    "What share of merits judgments after 2020 found a violation?",  # composed condition
    "How many cases involve an allegation of parental alienation?",
    "How long do proceedings take on average from application to judgment?",  # refused until 2026-07-07
    "In what proportion of cases did the mother receive custody?",
]
for q in DEMO:
    ask_anything(q)

Q: When can custody be transferred to the other parent because of alienating behaviour?
ROUTE: Bucket 1 (no aggregate trigger) -> retrieval + grounded generation
  ANSWER: The ECHR states that a transfer of a child from one parent to another by means of coercion aimed at breaking the child's resistance could not be justified if it is against the child's best interests and less restrictive measures suitable to achieve the legitimate objective in question were not seriously considered first.  AT (OGH) [2]: The District Court ordered a second expert in child psychology, Dr R., to submit an opinion on whether or not K.'s mother was capable of taking care of him. However, no information is provided on when custody can be transferred to the other parent due to alienat
  [1] cos=0.845 | ECHR      | CASE OF X AND OTHERS v. SLOVENIA
  [2] cos=0.842 | ECHR      | CASE OF SPORER v. AUSTRIA
  [3] cos=0.841 | ECHR      | CASE OF I.S. AND OTHERS v. MALTA
Q: Unter welchen Voraussetzungen kann einem E

In [10]:
ask_anything("How many contact_access cases are there aganist Romania?")

Q: How many contact_access cases are there aganist Romania?
ROUTE: aggregate trigger 'How many' -> query layer (never generation)
  BUCKET 2 (metadata aggregate) -- generated SQL (REVIEW THIS, it is the trust boundary):
    SELECT COUNT(*) AS cases FROM echr WHERE respondent_state = 'ROU' AND primary_theme = 'contact_access';
 cases
     5
  CAVEAT: counts describe the keyword-matched corpora, never litigation rates; citable numbers come from the reviewed canned queries in echr_query.ipynb.
